# 01 — Fetch a Paper by DOI via OpenAlex

**OpenAlex** is a fully open catalogue of the global research system. It indexes over 250 million scholarly works and provides a free REST API — no API key required.

This notebook shows how to:
1. Load the example paper dataset
2. Fetch full metadata for a single paper using its DOI
3. Explore the response fields
4. Extract the most useful fields into a flat dictionary

**API docs:** https://docs.openalex.org/api-entities/works/get-a-single-work

In [1]:
import requests
import pandas as pd
import json

## 1. Load example papers

In [2]:
df = pd.read_csv('../data/example_papers.csv')
print(f'Loaded {len(df)} papers')
df[['Paper id', 'DOI', 'Title', 'Inclusion decision']].head(10)

Loaded 10 papers


,Paper id,DOI,Title,Inclusion decision
0,4,10.3390/s23031255,An Efficient Machine Learning-Based Emotional ...,include
1,5,10.1038/s41598-023-35795-0,Machine learning explainability in nasopharyng...,include
2,6,10.1016/j.ymeth.2023.08.008,Machine learning algorithms for prediction of ...,include
3,7,10.1186/s12874-023-01837-4,Machine learning for predicting neurodegenerat...,include
4,8,10.1038/s41598-023-31542-7,Explanatory predictive model for COVID-19 seve...,include
5,11,10.1155/2023/9738123,Monitoring Cardiovascular Problems in Heart Pa...,include
6,1,10.1186/s12859-023-05235-x,A review and comparative study of cancer detec...,exclude
7,2,10.1371/journal.pone.0289348,Comparison of the data mining and machine lear...,exclude
8,3,10.1007/s11356-023-27387-2,Modelling and optimisation of electrocoagulati...,exclude
9,9,10.1007/s11356-023-26779-8,Improving the accuracy of air relative humidit...,exclude


## 2. Fetch one paper from OpenAlex by DOI

OpenAlex identifies works by DOI using the URL pattern:
```
https://api.openalex.org/works/https://doi.org/<DOI>
```

It is good practice to pass a `mailto` parameter so OpenAlex can contact you if your requests cause issues (this also grants you access to the faster "polite" pool).

In [3]:
MAILTO = 'your.email@example.com'  # replace with your e-mail

# Pick the first included paper
example_doi = df[df['Inclusion decision'] == 'include']['DOI'].iloc[0]
print('DOI:', example_doi)

DOI: 10.3390/s23031255


In [4]:
url = f'https://api.openalex.org/works/https://doi.org/{example_doi}'
params = {'mailto': MAILTO}

response = requests.get(url, params=params)
print('Status code:', response.status_code)

work = response.json()
print('OpenAlex ID:', work.get('id'))

Status code: 200
OpenAlex ID: https://openalex.org/W4317727959


## 3. Explore the raw response

In [5]:
# Top-level keys in the response
print('Top-level keys:')
for key in work.keys():
    val = work[key]
    val_preview = str(val)[:80] if val is not None else 'None'
    print(f'  {key}: {val_preview}')

Top-level keys:
  id: https://openalex.org/W4317727959
  doi: https://doi.org/10.3390/s23031255
  title: An Efficient Machine Learning-Based Emotional Valence Recognition Approach Towar
  display_name: An Efficient Machine Learning-Based Emotional Valence Recognition Approach Towar
  publication_year: 2023
  publication_date: 2023-01-21
  ids: {'openalex': 'https://openalex.org/W4317727959', 'doi': 'https://doi.org/10.3390
  language: en
  primary_location: {'id': 'doi:10.3390/s23031255', 'is_oa': True, 'landing_page_url': 'https://doi.
  type: article
  indexed_in: ['crossref', 'doaj', 'pubmed']
  open_access: {'is_oa': True, 'oa_status': 'gold', 'oa_url': 'https://www.mdpi.com/1424-8220/2
  authorships: [{'author_position': 'first', 'author': {'id': 'https://openalex.org/A5055186980
  institutions: []
  countries_distinct_count: 1
  institutions_distinct_count: 1
  corresponding_author_ids: ['https://openalex.org/A5055186980']
  corresponding_institution_ids: ['https://openalex.org/I

In [6]:
# Pretty-print a specific sub-object to understand its structure
print('Open access info:')
print(json.dumps(work.get('open_access'), indent=2))

Open access info:
{
  "is_oa": true,
  "oa_status": "gold",
  "oa_url": "https://www.mdpi.com/1424-8220/23/3/1255/pdf?version=1674301019",
  "any_repository_has_fulltext": true
}


In [7]:
print('Author list (first 3):')
print(json.dumps(work.get('authorships', [])[:3], indent=2))

Author list (first 3):
[
  {
    "author_position": "first",
    "author": {
      "id": "https://openalex.org/A5055186980",
      "display_name": "Lamiaa Abdel\u2010Hamid",
      "orcid": "https://orcid.org/0000-0002-4066-6021"
    },
    "institutions": [
      {
        "id": "https://openalex.org/I47853400",
        "display_name": "Misr International University",
        "ror": "https://ror.org/030vg1t69",
        "country_code": "EG",
        "type": "education",
        "lineage": [
          "https://openalex.org/I47853400"
        ]
      }
    ],
    "countries": [
      "EG"
    ],
    "is_corresponding": true,
    "raw_author_name": "Lamiaa Abdel-Hamid",
    "raw_affiliation_strings": [
      "Department of Electronics & Communication, Faculty of Engineering, Misr International University (MIU), Heliopolis, Cairo P.O. Box 1 , Egypt"
    ],
    "affiliations": [
      {
        "raw_affiliation_string": "Department of Electronics & Communication, Faculty of Engineering, Misr

In [8]:
print('Concepts (topics) assigned by OpenAlex:')
print(json.dumps(work.get('concepts', [])[:5], indent=2))

Concepts (topics) assigned by OpenAlex:
[
  {
    "id": "https://openalex.org/C522805319",
    "wikidata": "https://www.wikidata.org/wiki/Q179965",
    "display_name": "Electroencephalography",
    "level": 2,
    "score": 0.8122310638427734
  },
  {
    "id": "https://openalex.org/C41008148",
    "wikidata": "https://www.wikidata.org/wiki/Q21198",
    "display_name": "Computer science",
    "level": 0,
    "score": 0.707916259765625
  },
  {
    "id": "https://openalex.org/C154945302",
    "wikidata": "https://www.wikidata.org/wiki/Q11660",
    "display_name": "Artificial intelligence",
    "level": 1,
    "score": 0.6308228373527527
  },
  {
    "id": "https://openalex.org/C206310091",
    "wikidata": "https://www.wikidata.org/wiki/Q750859",
    "display_name": "Emotion classification",
    "level": 2,
    "score": 0.592275857925415
  },
  {
    "id": "https://openalex.org/C66905080",
    "wikidata": "https://www.wikidata.org/wiki/Q17005494",
    "display_name": "Binary classificatio

## 4. Extract key fields into a flat dictionary

In [9]:
def extract_work_fields(work: dict) -> dict:
    """Extract the most useful fields from an OpenAlex work object."""
    # Abstract: OpenAlex stores abstracts as an inverted index
    abstract = None
    if work.get('abstract_inverted_index'):
        index = work['abstract_inverted_index']
        words = [''] * (max(pos for positions in index.values() for pos in positions) + 1)
        for word, positions in index.items():
            for pos in positions:
                words[pos] = word
        abstract = ' '.join(words)

    authorships = work.get('authorships', [])
    authors = '; '.join(
        a['author']['display_name']
        for a in authorships
        if a.get('author')
    )
    institutions = '; '.join(
        inst['display_name']
        for a in authorships
        for inst in a.get('institutions', [])
        if inst.get('display_name')
    )
    top_concepts = ', '.join(
        c['display_name']
        for c in sorted(work.get('concepts', []), key=lambda x: -x.get('score', 0))[:5]
    )

    return {
        'openalex_id': work.get('id'),
        'doi': work.get('doi'),
        'title': work.get('title'),
        'publication_year': work.get('publication_year'),
        'publication_date': work.get('publication_date'),
        'type': work.get('type'),
        'cited_by_count': work.get('cited_by_count'),
        'is_oa': work.get('open_access', {}).get('is_oa'),
        'oa_status': work.get('open_access', {}).get('oa_status'),
        'authors': authors,
        'institutions': institutions,
        'concepts': top_concepts,
        'abstract': abstract,
    }


extracted = extract_work_fields(work)
for k, v in extracted.items():
    val = str(v)[:120] if v else v
    print(f'{k:25s}: {val}')

openalex_id              : https://openalex.org/W4317727959
doi                      : https://doi.org/10.3390/s23031255
title                    : An Efficient Machine Learning-Based Emotional Valence Recognition Approach Towards Wearable EEG
publication_year         : 2023
publication_date         : 2023-01-21
type                     : article
cited_by_count           : 25
is_oa                    : True
oa_status                : gold
authors                  : Lamiaa Abdel‐Hamid
institutions             : Misr International University
concepts                 : Electroencephalography, Computer science, Artificial intelligence, Emotion classification, Binary classification
abstract                 : Emotion artificial intelligence (AI) is being increasingly adopted in several industries such as healthcare and educatio


In [10]:
# Display as a single-row DataFrame
pd.DataFrame([extracted]).T.rename(columns={0: 'value'})

,value
openalex_id,https://openalex.org/W4317727959
doi,https://doi.org/10.3390/s23031255
title,An Efficient Machine Learning-Based Emotional ...
publication_year,2023
publication_date,2023-01-21
type,article
cited_by_count,25
is_oa,True
oa_status,gold
authors,Lamiaa Abdel‐Hamid


## 5. Compare OpenAlex citation count vs. Dimensions export

Our example CSV includes `Times cited` from Dimensions. Let's compare.

In [11]:
row = df[df['DOI'] == example_doi].iloc[0]
print(f"Title          : {row['Title']}")
print(f"Dimensions cit.: {row['Times cited']}")
print(f"OpenAlex cit.  : {extracted['cited_by_count']}")

Title          : An Efficient Machine Learning-Based Emotional Valence Recognition Approach Towards Wearable EEG
Dimensions cit.: 7
OpenAlex cit.  : 25
